In [39]:
# imports

import os 
import requests 
import pandas as pd
import numpy as np
import json

In [40]:
# Vancouver bike stations from part 1: 

van_bikes = pd.read_csv('C:/Users/sherr/LHL classwork/LHL-P2/data/van_bikes.csv')

# Foursquare

In [41]:
# Category IDs:

restaurant_id = '4d4b7105d754a06374d81259'
pilates_id = '5744ccdfe4b0c0459246b4b2'

In [42]:
url = 'https://api.foursquare.com/v3/places/search?'
api_key = os.environ['FOURSQUARE_API_KEY']
headers = {'Accept': 'application/json'}
headers['Authorization'] = api_key

result = requests.get('https://api.foursquare.com/v3/places/search?', headers=headers, )

Send a request to Foursquare with a small radius (1000m) for all the bike stations in your city of choice. 

In [43]:
url = 'https://api.foursquare.com/v3/places/search?'
api_key = os.environ['FOURSQUARE_API_KEY']
headers = {'Accept': 'application/json'}
headers['Authorization'] = api_key

fsq_restaurants = []
fsq_pilates = []

for row in van_bikes.itertuples(index=False):
    
    lat = row.latitude
    long = row.longitude 
    ll = f'{lat},{long}'
    
    r_params = {'ll': ll,
                'radius': 1000,
                'categories': restaurant_id}
    
    w_params = {'ll': ll, 
              'radius': 1000,
              'categories': pilates_id}
    
    r_result = requests.get(url, headers=headers, params=r_params)
    r_data = r_result.json()
    r_df = pd.json_normalize(r_data['results'])
    r_df = r_df.rename(columns={'geocodes.main.latitude': 'latitude',
                                'geocodes.main.longitude': 'longitude',
                                'location.address': 'address',
                                'location.country': 'country',
                                'location.locality': 'locality',
                                'location.postcode': 'postcode'})
    fsq_restaurants.append(r_df)

    w_result = requests.get(url, headers=headers, params=w_params)
    w_data = w_result.json()
    w_df = pd.json_normalize(w_data['results'])
    w_df = w_df.rename(columns={'geocodes.main.latitude': 'latitude',
                                'geocodes.main.longitude': 'longitude',
                                'location.address': 'address',
                                'location.country': 'country',
                                'location.locality': 'locality',
                                'location.postcode': 'postcode'})
    fsq_pilates.append(w_df)

In [46]:
yelp_pilates[1]

,name,address,distance,reviews,rating,price
0,Pilates Process - Vancouver,365 East Broadway,324.754810,1,5.0,None
1,Union Pilates,3381 Fraser Street,863.088640,0,0.0,None
2,Lagree West,199 E 20th Street,1204.871508,1,5.0,None
3,Evolution Fitness and Lagree,1143 Kingsway,1381.878329,3,5.0,None
4,Club Pilates,1119 Kingsway,1348.220911,0,0.0,None
5,Vancouver Corporate Yoga,1055 W Georgia Street,3294.199973,7,4.4,None
6,Mount Pleasant Community Centre,1 Kingsway,555.326161,36,3.1,None
7,i-Training Studio,208-2520 Ontario Street,842.497202,0,0.0,None
8,CMMN GRND Fitness and Social Wellness Collective,121 W 2nd Ave,1304.948551,5,2.8,None


Parse through the response to get the POI (such as restaurants, bars, etc) details you want (ratings, name, location, etc)

In [57]:
nearest_rdists = []
nearest_rnames = []
furthest_rdists = []
furthest_rnames = []
avg_rdists = []
nearest_wdists = []
nearest_wnames = []
furthest_wdists = []
furthest_wnames = []
avg_wdists = []
nearby_workouts = []

for i, row in van_bikes.iterrows():
    r_df = restaurants[i]
    nearest_rdist = r_df['distance'].min()
    nearest_rname = r_df.sort_values('distance').iloc[0]['name']
    furthest_rdist = r_df['distance'].max()
    furthest_rname = r_df.sort_values('distance', ascending=False).iloc[0]['name']
    avg_rdist = round((sum(list(r_df['distance'])) / len(r_df)), 2)

    w_df = workouts[i]
    if 'distance' in w_df.columns:
        nearest_wdist = w_df['distance'].min()
        furthest_wdist = w_df['distance'].max()
        avg_wdist = round(sum(list(w_df['distance'])) / len(w_df), 2)
    else:
        nearest_wdist = None
        furthest_wdist = None
        avg_wdist = None
        
    if 'name' in w_df.columns: 
        nearest_wname = w_df.sort_values('distance').iloc[0]['name']
        furthest_wname = w_df.sort_values('distance', ascending=False).iloc[0]['name']
    else:
        nearest_wname = None
        furthest_wname = None

    workouts_nearby = len(w_df)

    nearest_rdists.append(nearest_rdist)
    nearest_rnames.append(nearest_rname)
    furthest_rdists.append(furthest_rdist)
    furthest_rnames.append(furthest_rname)
    avg_rdists.append(avg_rdist)
    nearest_wdists.append(nearest_wdist)
    nearest_wnames.append(nearest_wname)
    furthest_wdists.append(furthest_wdist)
    furthest_wnames.append(furthest_wname)
    avg_wdists.append(avg_wdist)
    nearby_workouts.append(workouts_nearby)

Put your parsed results into a DataFrame

In [58]:
foursquare_df = pd.DataFrame({'name': van_bikes['name'],
              'timestamp': van_bikes['timestamp'],
              'free_bikes': van_bikes['free_bikes'],
              'empty_slots': van_bikes['empty_slots'],
              'nearest_rdist': nearest_rdists,
              'nearest_rname': nearest_rnames,
              'furthest_rdist': furthest_rdists,
              'furthest_rname': furthest_rnames,
              'avg_rdist': avg_rdists,
              'nearest_wdist': nearest_wdists,
              'nearest_wname': nearest_wnames,
              'furthest_wdist': furthest_wdists,
              'furthest_wname': furthest_wnames,
              'avg_wdist': avg_wdists,
              'nearby_workouts': nearby_workouts})

In [60]:
foursquare_df.head()

,name,timestamp,free_bikes,empty_slots,nearest_rdist,nearest_rname,furthest_rdist,furthest_rname,avg_rdist,nearest_wdist,nearest_wname,furthest_wdist,furthest_wname,avg_wdist,nearby_workouts
0,Chilco & Barclay,2025-04-17T00:24:45.486341Z,5,13,308,Kingyo Izakaya 金魚居酒屋,869,The Inukshuk,518.7,366.0,Oxygen Yoga West End,862.0,Royal Private Coach,649.00,5
1,St George & Broadway,2025-04-17T00:24:45.944688Z,0,14,341,12 Kings Pub,906,33 Acres Brewing Co,615.0,329.0,Vancouver Pilates Process Inc,940.0,Dharma Temple,704.50,10
2,Britannia Parking Lot,2025-04-17T00:24:45.920383Z,0,14,160,The Downlow Chicken Shack,506,Via Tevere Pizzeria,336.3,201.0,Bikram Yoga Commercial Drive,836.0,Dharmalab,442.00,4
3,Morton & Denman,2025-04-17T00:24:45.074554Z,21,4,89,Bayside Lounge,668,Hokkaido Ramen Santouka,297.2,537.0,Oxygen Yoga West End,981.0,Royal Private Coach,735.75,4
4,Thornton & National,2025-04-17T00:24:45.920601Z,9,5,552,Torafuku,902,Keefer Bar,742.8,675.0,Radha Vancouver,917.0,Stretch,775.00,3


# Yelp

Send a request to Yelp with a small radius (1000m) for all the bike stations in your city of choice. 

In [4]:
url = 'https://api.yelp.com/v3/businesses/search?categories=&sort_by=best_match&limit=10'
api_key = os.environ["YELP_API_KEY"]
headers = {'Accept': 'application/json',
           'Authorization': f'Bearer {api_key}'}

yelp_restaurants = []
yelp_pilates = []

for row in van_bikes.itertuples(index=False):
    latitude = row.latitude
    longitude = row.longitude

    r_params = {'latitude': latitude,
                'longitude': longitude,
                'radius': 1000,
                'term': 'restaurant'}
    
    p_params = {'latitude': latitude,
                'longitude': longitude,
                'radius': 1000,
                'term': 'pilates'}   
    
    r_result = requests.get(url, headers=headers, params=r_params)
    r_data = r_result.json()['businesses']
    r_columns = [{'name': r['name'],
                 'address': r['location']['address1'],
                 'distance': r['distance'],
                 'reviews': r['review_count'],
                 'rating': r['rating'],
                 'price': r.get('price')} for r in r_data]
    r_df = pd.DataFrame(r_columns)
    yelp_restaurants.append(r_df)

    p_result = requests.get(url, headers=headers, params=p_params)
    p_data = p_result.json()['businesses']
    p_columns = [{'name': p['name'],
                 'address': p['location']['address1'],
                 'distance': p['distance'],
                 'reviews': p['review_count'],
                 'rating': p['rating'],
                 'price': p.get('price')} for p in p_data]
    p_df = pd.DataFrame(p_columns)   
    yelp_pilates.append(p_df)

    

Parse through the response to get the POI (such as restaurants, bars, etc) details you want (ratings, name, location, etc)

In [36]:
# since 4sq already got nearest/furthest restaurants, info I want for yelp is: 
# avg rating of r/p in radius and avg proximity (avg distance), best rated(if tie then more reviews) and most reviewed 

avg_rratings = []
avg_rdists = []
best_rratingnames = []
best_rratings = []
best_rdists = []


avg_pratings = []
avg_pdists = []
best_pratingnames = []
best_pratings = []
best_pdists = []

for i, row in van_bikes.iterrows():
    
    r_df = yelp_restaurants[i]
    avg_rrating = round(r_df['rating'].mean(), 2)
    avg_rdist = round(r_df['distance'].mean(), 2)
    best_rratingname = r_df.sort_values(['rating', 'reviews'], ascending=[False, False])['name'].values[0]
    best_rrating = r_df.sort_values(['rating', 'reviews'], ascending=[False, False])['rating'].values[0]
    best_rdist = r_df.sort_values(['rating', 'reviews'], ascending=[False, False])['distance'].values[0]

    p_df = yelp_pilates[i]
    if 'rating' in p_df.columns and 'distance' in p_df.columns and 'reviews' in p_df.columns:
        avg_prating = round(p_df['rating'].mean(), 2)
        avg_pdist = round(p_df['distance'].mean(), 2)
        best_pratingname = p_df.sort_values(['rating', 'reviews'], ascending=[False, False])['name'].values[0]
        best_prating = p_df.sort_values(['rating', 'reviews'], ascending=[False, False])['rating'].values[0]
        best_pdist = p_df.sort_values(['rating', 'reviews'], ascending=[False, False])['distance'].values[0]

    else:
        avg_prating = None
        avg_pdist = None
        best_pratingname = None
        best_prating = None
        best_pdist = None 

    avg_rratings.append(avg_rrating)
    avg_rdists.append(avg_rdist)
    best_rratingnames.append(best_rratingname)
    best_rratings.append(best_rrating)
    best_rdists.append(best_rdist)
    
    avg_pratings.append(avg_prating)
    avg_pdists.append(avg_pdist)
    best_pratingnames.append(best_pratingname)
    best_pratings.append(best_prating)
    best_pdists.append(best_pdist)
   

In [37]:
yelp_df = pd.DataFrame({'station_name': van_bikes['name'],
                        'timestamp': van_bikes['timestamp'],
                        'free_bikes': van_bikes['free_bikes'],
                        'empty_slots': van_bikes['empty_slots'],
                        'avg_rrating': avg_rratings,
                        'avg_rdist': avg_rdists,
                        'best_rratingname': best_rratingnames,
                        'best_rrating': best_rratings,
                        'best_rdist': best_rdists,
                        'avg_prating': avg_pratings,
                        'avg_pdist': avg_pdists,
                        'best_pratingname': best_pratingnames,
                        'best_prating': best_pratings,
                        'best_pdist': best_pdists})

yelp_df.to_csv('C:/Users/sherr/LHL classwork/LHL-P2/data/yelp_df.csv', index=False)

Put your parsed results into a DataFrame

In [38]:
yelp_rdf = pd.concat(yelp_restaurants, ignore_index=True)
yelp_pdf = pd.concat(yelp_pilates, ignore_index=True)
yelp_pdf.head()

,name,address,distance,reviews,rating,price
0,Pilates Unlimited Fitness & Rehabilitation Studio,1706 Alberni Street,545.485703,1,3.0,None
1,The Well by Kunye,1579 W Georgia Street,777.755119,1,5.0,None
2,YYOGA Downtown Flow,888 Burrard Street,1628.518110,128,3.7,None
3,Wholistic Osteopathy And Wellness,960-1111 Melville Street,1465.687502,1,4.0,None
4,Club Pilates,941 Hornby Street,1688.927117,3,2.3,None


# Comparing Results

Which API provided you with more complete data? Provide an explanation. 

Yelp API provided me with more complete data, because it is able to show all the information that Foursquare shows in addition to showing the rating and number of reviews.

Get the top 10 restaurants according to their rating

In [23]:
yelp_rdf.sort_values('rating', ascending=False).head(10)

,name,address,distance,reviews,rating,price
886,Nero Waffles,2861 Granville Street,571.729634,2,5.0,None
969,Uncle Pu's Sichuan Taste,3132 West Broadway,571.130714,2,5.0,None
1014,Magari by Oca,1260 Commercial Drive,222.782372,3,5.0,None
829,Seoul Hotdog,6133 University Boulevard,525.993721,1,5.0,None
838,Magari by Oca,1260 Commercial Drive,855.622904,3,5.0,None
1573,Word.,555 Great Northern Way,206.110225,1,5.0,None
1554,Kapow Burger,20 East 4th Avenue,135.191785,1,5.0,None
1117,Desi Indian Lounge,1355 Hornby Street,492.488775,2,5.0,None
1182,Word.,555 Great Northern Way,88.175613,1,5.0,None
1275,Uncle Pu's Sichuan Taste,3132 West Broadway,1219.403566,2,5.0,None
